# Retail Business Performance Analysis
## Superstore Dataset — United States

### Objective
To identify if the business is growing or not and if not then what are the factors which affects the business. To do this we need to find which regions, categories, segments and products are driving profit and which are destroying value — and why.

### Tools Used
Python, Pandas, SQL, Power BI

### Dataset
10194 rows | 21 columns 

## 1. Data Loading & Cleaning

In [1]:
import pandas as pd
df = pd.read_csv('sample_-_superstore.csv')
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,US-2023-103800,01/03/2023,01/07/2023,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,...,77095,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",16.448,2,0.2,5.5512
1,2,US-2023-112326,01/04/2023,01/08/2023,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.540,2,0.8,-5.4870
2,3,US-2023-112326,01/04/2023,01/08/2023,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784,3,0.2,4.2717
3,4,US-2023-112326,01/04/2023,01/08/2023,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,272.736,3,0.2,-64.7748
4,5,US-2023-141817,01/05/2023,01/12/2023,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,...,19143,East,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,19.536,3,0.2,4.8840


In [2]:
#rows and columns
df.shape

(10194, 21)

In [3]:
#checking for missing values
df.isnull().sum()

Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Customer Name     0
Segment           0
Country/Region    0
City              0
State/Province    0
Postal Code       0
Region            0
Product ID        0
Category          0
Sub-Category      0
Product Name      0
Sales             0
Quantity          0
Discount          0
Profit            0
dtype: int64

In [4]:
#data types
df.dtypes

Row ID              int64
Order ID           object
Order Date         object
Ship Date          object
Ship Mode          object
Customer ID        object
Customer Name      object
Segment            object
Country/Region     object
City               object
State/Province     object
Postal Code        object
Region             object
Product ID         object
Category           object
Sub-Category       object
Product Name       object
Sales             float64
Quantity            int64
Discount          float64
Profit            float64
dtype: object

### We need to change the data type of order date and ship date

In [5]:
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])
df.dtypes

Row ID                     int64
Order ID                  object
Order Date        datetime64[ns]
Ship Date         datetime64[ns]
Ship Mode                 object
Customer ID               object
Customer Name             object
Segment                   object
Country/Region            object
City                      object
State/Province            object
Postal Code               object
Region                    object
Product ID                object
Category                  object
Sub-Category              object
Product Name              object
Sales                    float64
Quantity                   int64
Discount                 float64
Profit                   float64
dtype: object

## 2. Regional analysis

### How each region is performing?

In [6]:
#sales sum
df.groupby('Region')['Sales'].sum().sort_values(ascending=False)

Region
West       739813.6085
East       691828.1680
Central    503170.6728
South      391721.9050
Name: Sales, dtype: float64

In [7]:
#profit sum
df.groupby('Region')['Profit'].sum().sort_values(ascending=False)

Region
West       110798.8170
East        94883.2603
South       46749.4303
Central     39865.3070
Name: Profit, dtype: float64

In [8]:
# Profit margin — which region has high sales but low profit?
region = df.groupby('Region')[['Sales','Profit']].sum()
region['Profit_Margin'] = (region['Profit'] / region['Sales']) * 100
region.round(2)

,Sales,Profit,Profit_Margin
Region,,,
Central,503170.67,39865.31,7.92
East,691828.17,94883.26,13.71
South,391721.90,46749.43,11.93
West,739813.61,110798.82,14.98


### Insights
Central region generates high sales (3rd highest) but has the worst profit margin at 7.92%. This suggests heavy discounting or high costs are eating into profits despite decent revenue.

East and West regions are doing good and south has low sales but a healthier profit margin than central.

## 3. Deep Analysis of Central Region - Why central region is not doing good ?
Is Central being over discounted?

In [9]:
df.groupby('Region')['Discount'].mean().sort_values(ascending=False)

Region
Central    0.241002
South      0.147253
East       0.143436
West       0.108946
Name: Discount, dtype: float64

As we can see central region gives more discount.
Central Region gives the highest discounts (24%) AND has the worst profit margin. South Region gives second highest discounts but still maintains decent profit.

### Category & Sub-Category Analysis - Central Region

In [23]:
#Category
df[df['Region']=='Central'].groupby('Category')[['Sales','Profit']].sum().round(2)

,Sales,Profit
Category,,
Furniture,164537.65,-2802.21
Office Supplies,168216.71,8970.08
Technology,170416.31,33697.43


Central region is being carried entirely by Technology. 
Furniture is actively losing money despite high sales. 
Office Supplies margin is dangerously thin at 5.3%.
The heavy discounting (24% average) is likely destroying 
Furniture and Office Supplies profitability.

In [24]:
#discount on categories
df[df['Region']=='Central'].groupby('Category')['Discount'].mean().round(3)

Category
Furniture          0.297
Office Supplies    0.254
Technology         0.133
Name: Discount, dtype: float64

"Central's losses are driven by excessive discounting on Furniture and Office Supplies — averaging 29.7% and 25.4% respectively — while Technology's controlled 13.3% discount rate makes it the only profitable category in the region."

Note: This dataset does not include production costs, 
marketing spend, or campaign data. Therefore we can 
identify WHERE the problem is but cannot conclusively 
determine the root cause without additional data sources.

In [25]:
#Sub-Categories & Discount on sub-categories
df[df['Region']=='Central'].groupby('Sub-Category')[['Sales','Profit','Discount']].agg({
    'Sales':'sum',
    'Profit':'sum',
    'Discount':'mean'
}).round(3).sort_values('Profit')

,Sales,Profit,Discount
Sub-Category,,,
Furnishings,15284.362,-3915.515,0.404
Tables,39154.971,-3559.650,0.262
Appliances,23582.033,-2638.618,0.449
Bookcases,24157.177,-1997.904,0.233
Machines,26797.384,-1486.067,0.329
Binders,58060.790,-958.305,0.509
Supplies,9467.372,-661.888,0.144
Fasteners,778.030,236.619,0.135
Labels,2451.472,1073.079,0.113


Central region's losses are entirely driven by excessive 
discounting. Any sub-category discounted above 25-30% 
consistently loses money. The business should cap discounts 
at 20% maximum for Furniture and Office Supplies in Central.

## 4. Category analysis of all regions

In [13]:
df.groupby(['Region','Category'])[['Sales','Profit','Discount']].agg({'Sales':'sum',
    'Profit':'sum',
    'Discount':'mean'
}).round(3)

Sales     Profit  Discount
Region  Category                                        
Central Furniture        164537.652  -2802.207     0.297
        Office Supplies  168216.709   8970.082     0.254
        Technology       170416.312  33697.432     0.133
East    Furniture        212231.696   3444.745     0.154
        Office Supplies  211658.401  42996.740     0.140
        Technology       267938.071  48441.776     0.141
South   Furniture        117298.684   6771.206     0.122
        Office Supplies  125651.313  19986.393     0.167
        Technology       148771.908  19991.831     0.108
West    Furniture        260679.730  12316.251     0.130
        Office Supplies  226366.891  54070.229     0.093
        Technology       252766.988  44412.336     0.133

### Insight:
Furniture is a structurally low margin category across ALL regions 
— not just Central. Even in the best case (South at 5.7%), it 
barely justifies the shelf space. Central's heavy discounting 
pushes it into loss territory entirely.

This suggests the problem is not just a regional discount policy 
issue — Furniture may simply be an unprofitable category for 
this business regardless of region.

## 5.Yearly analysis

In [14]:
# Extract year and month from Order Date
df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month

# Yearly sales and profit
df.groupby('Year')[['Sales','Profit']].sum().round(2)

,Sales,Profit
Year,,
2023,494040.21,51684.30
2024,472993.03,62020.97
2025,613933.58,82665.20
2026,745567.53,95926.35


In [20]:
# yearly profit margin
yearly = df.groupby('Year')[['Sales','Profit']].sum().round(2)
yearly['Profit_Margin'] = (yearly['Profit'] / yearly['Sales'] * 100).round(2)
yearly

,Sales,Profit,Profit_Margin
Year,,,
2023,494040.21,51684.30,10.46
2024,472993.03,62020.97,13.11
2025,613933.58,82665.20,13.46
2026,745567.53,95926.35,12.87


### Insight:
Revenue grew 50% from 2023 to 2026 but profit margin 
remained flat between 10-13%. The business is scaling 
in size but not in profitability. Growth is not 
translating into better margins — suggesting the 
underlying discount and cost problems identified in 
Central are a persistent company-wide issue.

## 6.Segment analysis

In [16]:
df.groupby('Segment')[['Sales','Profit']].agg({
    'Sales':'sum',
    'Profit':'sum'
}).round(2)

,Sales,Profit
Segment,,
Consumer,1170659.79,136371.45
Corporate,715806.13,94249.64
Home Office,440068.43,61675.73


By looking at the numbers consumer segment has high sales and home office has low sales but see in next cell which segment is giving business more profit

In [17]:
segment = df.groupby('Segment')[['Sales','Profit']].sum()
segment['Profit_Margin'] = (segment['Profit'] / segment['Sales'] * 100).round(2)
segment

,Sales,Profit,Profit_Margin
Segment,,,
Consumer,1.170660e+06,136371.4463,11.65
Corporate,7.158061e+05,94249.6400,13.17
Home Office,4.400684e+05,61675.7283,14.02


### Insight:
Home Office is the smallest segment by revenue but the most 
profitable by margin at 14.02%. Consumer drives the most 
revenue but has the lowest margin at 11.65% — likely due to 
higher discounting to attract mass market customers. 
Corporate sits in the middle on both metrics.

## 7.Product analysis

In [18]:
# Top 10 profitable products
df.groupby('Product Name')['Profit'].sum().sort_values(ascending=True).head(10)

Product Name
Cubify CubeX 3D Printer Double Head Print                           -8879.9704
Lexmark MX611dhe Monochrome Laser Printer                           -4589.9730
Cubify CubeX 3D Printer Triple Head Print                           -3839.9904
Chromcraft Bull-Nose Wood Oval Conference Tables & Bases            -2876.1156
Bush Advantage Collection Racetrack Conference Table                -1934.3976
GBC DocuBind P400 Electric Binding System                           -1878.1662
Cisco TelePresence System EX90 Videoconferencing Unit               -1811.0784
Martin Yale Chadless Opener Electric Letter Opener                  -1299.1836
Balt Solid Wood Round Tables                                        -1201.0581
BoxOffice By Design Rectangular and Half-Moon Meeting Room Tables   -1148.4375
Name: Profit, dtype: float64

In [21]:
# Bottom 10 profitable products
df.groupby('Product Name')['Profit'].sum().sort_values(ascending=False).tail(10)

Product Name
BoxOffice By Design Rectangular and Half-Moon Meeting Room Tables   -1148.4375
Balt Solid Wood Round Tables                                        -1201.0581
Martin Yale Chadless Opener Electric Letter Opener                  -1299.1836
Cisco TelePresence System EX90 Videoconferencing Unit               -1811.0784
GBC DocuBind P400 Electric Binding System                           -1878.1662
Bush Advantage Collection Racetrack Conference Table                -1934.3976
Chromcraft Bull-Nose Wood Oval Conference Tables & Bases            -2876.1156
Cubify CubeX 3D Printer Triple Head Print                           -3839.9904
Lexmark MX611dhe Monochrome Laser Printer                           -4589.9730
Cubify CubeX 3D Printer Double Head Print                           -8879.9704
Name: Profit, dtype: float64

### Insight:
Top profit drivers are all Technology products — Copiers and 
Printers dominate. Bottom loss makers are dominated by 
Furniture (Tables) confirming earlier regional finding. 

## 8. Business Recommendations

1. Cap discounts at 20% maximum across all regions — 
   especially for Furniture and Office Supplies in Central
2. Review Furniture category pricing strategy company-wide — 
   it is structurally low margin in every region
3. Invest more in Technology — highest margin category consistently
4. Focus on Home Office segment — smallest but most efficient at 14% margin
5. Discontinue or reprice bottom loss-making products — 
   especially Tables and Binders in Central